In [3]:
import torch
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Using Apple Metal Performance Shaders (MPS)")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
#     print("Using CUDA GPU")
# else:
#     device = torch.device("cpu")
#     print("Using CPU")h
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 4080


In [4]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import (
    PreTrainedTokenizerFast,
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    Trainer,
    TrainingArguments
)
import json
from tqdm import tqdm

# ============ 1. Create proper tokenizer ============
tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordLevelTrainer(
    special_tokens=["<pad>", "<s>", "</s>", "<unk>"]
)

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Metal Performance Shaders (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")


# Train on space-separated tokens
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print("=" * 60)
print("TOKENIZER INFO")
print("=" * 60)
print(f"Vocab size: {len(tokenizer)}")
print(f"Token IDs: pad={tokenizer.pad_token_id}, bos={tokenizer.bos_token_id}, eos={tokenizer.eos_token_id}")
print(f"Vocabulary: {tokenizer.get_vocab()}")
print(f"Test encoding 'R U R U': {tokenizer.encode('R U R U')}")
print(f"Test decoding: '{tokenizer.decode(tokenizer.encode('R U R U'))}'")
print()

# ============ 2. Create model with correct vocab size ============
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-224",
    "gpt2"
)

# CRITICAL: Resize decoder vocabulary BEFORE setting configs
print("Resizing decoder vocabulary...")
model.decoder.resize_token_embeddings(len(tokenizer))

# Set special tokens everywhere they're needed
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.vocab_size = len(tokenizer)

model.decoder.config.bos_token_id = tokenizer.bos_token_id
model.decoder.config.eos_token_id = tokenizer.eos_token_id
model.decoder.config.pad_token_id = tokenizer.pad_token_id
model.decoder.config.vocab_size = len(tokenizer)

image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

print(f"Model decoder vocab size: {model.decoder.config.vocab_size}")
print(f"Model config decoder_start_token_id: {model.config.decoder_start_token_id}")
print()

# ============ 3. Dataset with proper formatting ============
class MazeDataset(Dataset):
    def __init__(self, entries, processor, tokenizer):
        self.entries = entries
        self.processor = processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["image"]).convert("RGB")

        pixel_values = self.processor(img, return_tensors="pt").pixel_values[0]
        
        # Join with spaces for WordLevel tokenizer
        sequence_text = " ".join(e["sequence"])
        labels = self.tokenizer(
            sequence_text,
            return_tensors="pt",
            padding=False,
            truncation=False
        ).input_ids[0]

        return {"pixel_values": pixel_values, "labels": labels}

# ============ 4. Custom collator ============
class MazeCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        
        labels = [item["labels"] for item in batch]
        max_len = max(len(l) for l in labels)
        
        # Pad labels
        padded_labels = []
        for label in labels:
            padded = torch.cat([
                label,
                torch.full((max_len - len(label),), self.pad_token_id, dtype=label.dtype)
            ])
            padded_labels.append(padded)
        
        labels = torch.stack(padded_labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# ============ 5. Load data ============
entries = json.load(open("data/grid_sequences.json"))
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
print(f"Total examples: {len(entries)}")
print(f"Sample entry: {entries[0]}")
print(f"Sample sequence: {' '.join(entries[0]['sequence'])}")
print()

dataset = MazeDataset(entries, image_processor, tokenizer)
collator = MazeCollator(tokenizer)

# Test dataset
sample = dataset[0]
print(f"Sample pixel_values shape: {sample['pixel_values'].shape}")
print(f"Sample labels: {sample['labels']}")
print(f"Decoded labels: '{tokenizer.decode(sample['labels'], skip_special_tokens=True)}'")
print()

model.to(device)

# ============ 6. Training with better settings ============
args = TrainingArguments(
    output_dir="maze-model",
    per_device_train_batch_size=2,  # Smaller batch for better gradients
    num_train_epochs=100,  # More epochs
    learning_rate=1e-4,  # Higher learning rate
    logging_steps=5,
    save_steps=500,
    save_total_limit=2,
    warmup_steps=10,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator
)

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Epochs: {args.num_train_epochs}")
print(f"Batch size: {args.per_device_train_batch_size}")
print(f"Learning rate: {args.learning_rate}")
print(f"Total steps: {len(dataset) // args.per_device_train_batch_size * args.num_train_epochs}")
print()

result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Final loss: {result.training_loss:.6f}")
print()

# ============ 7. Test inference with detailed output ============
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

print("=" * 60)
print("TESTING INFERENCE")
print("=" * 60)

correct = 0
total = min(5, len(entries))

for i in range(total):
    img = Image.open(entries[i]["image"])
    pixel_values = image_processor(img, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        # Generate with explicit parameters
        generated_ids = model.generate(
            pixel_values,
            max_length=30,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
            decoder_start_token_id=tokenizer.bos_token_id
        )
    
    predicted = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    expected = " ".join(entries[i]["sequence"])
    
    match = predicted == expected
    if match:
        correct += 1
    
    print(f"\nMaze {i}:")
    print(f"  Expected:  '{expected}'")
    print(f"  Predicted: '{predicted}'")
    print(f"  Match: {match}")
    print(f"  Generated IDs: {generated_ids[0].tolist()[:15]}")

print(f"\n{'=' * 60}")
print(f"Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
print(f"{'=' * 60}\n")

# Save model
model.save_pretrained("maze-model/final")
tokenizer.save_pretrained("maze-model/final")
print("Model saved to maze-model/final")

# ============ 8. Additional diagnostics ============
print("\n" + "=" * 60)
print("DIAGNOSTICS")
print("=" * 60)

# Check if model is actually using the small vocab
print(f"Decoder embedding weight shape: {model.decoder.transformer.wte.weight.shape}")
print(f"Expected shape: (6, 768)")  # 6 tokens, 768 hidden dim

# Test forward pass
with torch.no_grad():
    sample_batch = collator([dataset[0], dataset[1]])
    sample_batch = {k: v.to(device) for k, v in sample_batch.items()}
    outputs = model(**sample_batch)
    print(f"Logits shape: {outputs.logits.shape}")  # Should be (batch, seq_len, 6)
    print(f"Training loss on sample: {outputs.loss.item():.6f}")

Using CUDA GPU
TOKENIZER INFO
Vocab size: 6
Token IDs: pad=0, bos=1, eos=2
Vocabulary: {'<s>': 1, '</s>': 2, '<unk>': 3, 'R': 4, '<pad>': 0, 'U': 5}
Test encoding 'R U R U': [4, 5, 4, 5]
Test decoding: 'R U R U'



Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['transformer.h.0.crossattention.c_attn.bias', 'transformer.h.0.crossattention.c_attn.weight', 'transformer.h.0.crossattention.c_proj.bias', 'transformer.h.0.crossattention.c_proj.weight', 'transformer.h.0.crossattention.q_attn.bias', 'transformer.h.0.crossattention.q_attn.weight', 'transformer.h.0.ln_cross_attn.bias', 'transformer.h.0.ln_cross_attn.weight', 'transformer.h.1.crossattention.c_attn.bias', 'transformer.h.1.crossattention.c_attn.weight', 'transformer.h.1.crossattention.c_proj.bias', 'transformer.h.1.crossattention.c_proj.weight', 'transformer.h.1.crossattention.q_attn.bias', 'tran

Resizing decoder vocabulary...
Model decoder vocab size: 6
Model config decoder_start_token_id: 1

DATASET INFO
Total examples: 924
Sample entry: {'id': 0, 'sequence': ['R', 'R', 'R', 'R', 'R', 'R', 'U', 'U', 'U', 'U', 'U', 'U'], 'image': 'data/grids/grid_0.png'}
Sample sequence: R R R R R R U U U U U U

Sample pixel_values shape: torch.Size([3, 224, 224])
Sample labels: tensor([4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5])
Decoded labels: 'R R R R R R U U U U U U'

STARTING TRAINING
Epochs: 100
Batch size: 2
Learning rate: 0.0001
Total steps: 46200



Step,Training Loss
5,1.406800
10,1.172800
15,0.819400
20,0.664600
25,0.763500
30,0.766100
35,0.756800
40,0.697500
45,0.661200
50,0.675000



TRAINING COMPLETE
Final loss: 0.544206

TESTING INFERENCE

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R U R U R U R U U U U U U U R U U R U U U R U U R U U R'
  Match: False
  Generated IDs: [1, 4, 4, 5, 4, 5, 4, 5, 4, 5, 5, 5, 5, 5, 5]

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R U R U R U R U U U R U R U R U R U U R U R U R U U R U'
  Match: False
  Generated IDs: [1, 4, 4, 5, 4, 5, 4, 5, 4, 5, 5, 5, 4, 5, 4]

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R R U R U R U U U U U U U U U R U U R U U U R U U R U U'
  Match: False
  Generated IDs: [1, 4, 4, 4, 5, 4, 5, 4, 5, 5, 5, 5, 5, 5, 5]

Maze 3:
  Expected:  'R R R R R U U U R U U U'
  Predicted: 'R R R U U R U R U U U U U U U U U R U U R U U U R U U R U'
  Match: False
  Generated IDs: [1, 4, 4, 4, 5, 5, 4, 5, 4, 5, 5, 5, 5, 5, 5]

Maze 4:
  Expected:  'R R R R R U U U U R U U'
  Predicted: 'R R R U U R U R U U U R U R U U R U R U U R U U R U R U U'
  Match: False
  Generat

In [ ]:
from PIL import Image
import torch
from torch.utils.data import Dataset, Subset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import (
    PreTrainedTokenizerFast,
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    Trainer,
    TrainingArguments
)
import json

# ============ 1. Setup tokenizer ============
tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print("Vocab:", tokenizer.get_vocab())

# ============ 2. Create model ============
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-224",
    "gpt2"
)

model.decoder.resize_token_embeddings(len(tokenizer))
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.vocab_size = len(tokenizer)
model.decoder.config.vocab_size = len(tokenizer)

image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# ============ 3. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, processor, tokenizer):
        self.entries = entries
        self.processor = processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["image"]).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values[0]
        sequence_text = " ".join(e["sequence"])
        labels = self.tokenizer(sequence_text, return_tensors="pt").input_ids[0]
        return {"pixel_values": pixel_values, "labels": labels}

class MazeCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        labels = [item["labels"] for item in batch]
        max_len = max(len(l) for l in labels)
        padded_labels = []
        for label in labels:
            padded = torch.cat([
                label,
                torch.full((max_len - len(label),), self.pad_token_id, dtype=label.dtype)
            ])
            padded_labels.append(padded)
        labels = torch.stack(padded_labels)
        return {"pixel_values": pixel_values, "labels": labels}

# ============ 4. Load ONLY 10 examples ============
entries = json.load(open("data/grid_sequences.json"))
print(f"\nTotal dataset: {len(entries)} examples")
print(f"Using only first 10 for overfitting test\n")

# Use only first 10 examples
small_entries = entries[:10]
dataset = MazeDataset(small_entries, image_processor, tokenizer)
collator = MazeCollator(tokenizer)

# ============ 5. Train with MUCH lower LR and more epochs ============
args = TrainingArguments(
    output_dir="maze-model-debug",
    per_device_train_batch_size=2,
    num_train_epochs=200,  # More epochs
    learning_rate=5e-5,    # LOWER learning rate
    logging_steps=20,
    save_steps=5000,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator
)

print("=" * 60)
print("OVERFITTING TEST - Training on 10 examples only")
print("=" * 60)
print("If this doesn't reach near-zero loss, something is wrong!")
print()

result = trainer.train()

print(f"\nFinal loss: {result.training_loss:.6f}")
print("Expected: <0.01 for successful overfitting\n")

# ============ 6. Test on those same 10 examples ============
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("=" * 60)
print("Testing on the 10 training examples")
print("=" * 60)

correct = 0
for i in range(10):
    img = Image.open(small_entries[i]["image"])
    pixel_values = image_processor(img, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=30,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
        )
    
    predicted = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    expected = " ".join(small_entries[i]["sequence"])
    match = predicted == expected
    if match:
        correct += 1
    
    print(f"\nMaze {i}:")
    print(f"  Expected:  '{expected}'")
    print(f"  Predicted: '{predicted}'")
    print(f"  Match: {'✓' if match else '✗'}")

print(f"\n{'=' * 60}")
print(f"Accuracy: {correct}/10 ({100*correct/10:.0f}%)")
print(f"{'=' * 60}")

if correct >= 8:
    print("\n✓ Model CAN learn! Your setup is correct.")
    print("  → Solution: Train on full dataset with lower LR (5e-5)")
elif correct >= 3:
    print("\n⚠ Model is learning but slowly")
    print("  → Try even lower LR (1e-5) or more epochs (300+)")
else:
    print("\n✗ Model NOT learning - fundamental issue")
    print("  → Check: data format, tokenizer, model config")